# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

---

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id` values. We'll inspect the dataset and display all available record sets, fields, and columns.

In [ ]:
# List all record sets with their @id and name
print("Available Record Sets (by @id and name):")
record_sets = dataset.metadata.record_set
record_set_ids = []
for rs in record_sets:
    print(f"  @id: {rs.id}, name: {rs.name}")
    record_set_ids.append(rs.id)

# For each record set, show its fields and columns (using their @id fields)
all_fields_by_rs = {}
for rs in record_sets:
    print(f"\nFields for Record Set '{rs.name}' (@id: {rs.id}):")
    if hasattr(rs, "field") and rs.field:
        for field in rs.field:
            # Each field may contain column information
            column_ids = [col.id for col in getattr(field, "column", [])] if hasattr(field, "column") and field.column else []
            print(f"  Field: @id: {field.id}, name: {getattr(field, 'name', None)}, columns: {column_ids}")
        all_fields_by_rs[rs.id] = [f.id for f in rs.field]
    else:
        print("  (No fields listed)")

## 3. Data Extraction

Let's load records from a record set into a pandas DataFrame. We'll use the `@id` values obtained above to ensure we reference entities consistently.

In [ ]:
# We'll extract data from all record sets into DataFrames (referencing only by @id)
dataframes = {}
for recset_id in record_set_ids:
    # Dataset.records() yields dicts keyed by field @id
    print(f"Extracting records for record_set: {recset_id}")
    records = list(dataset.records(record_set=recset_id))
    dataframes[recset_id] = pd.DataFrame(records)
    print(f"  Columns: {dataframes[recset_id].columns.tolist()}")

# Display the head of the first available record set (by @id)
preview_rs_id = record_set_ids[0] if record_set_ids else None
if preview_rs_id:
    print(f"First few records of the record set '@id': {preview_rs_id}")
    display(dataframes[preview_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping/categorizing data.

**Note:** Make sure to reference the columns and fields by their `@id` properties, as listed above.


In [ ]:
# For demonstration, select numeric and grouping fields by @id for the first available record set
from pandas.api.types import is_numeric_dtype

# Pick a record set
record_set_id = record_set_ids[0] if record_set_ids else None

df = dataframes[record_set_id] if record_set_id else None

if df is not None and not df.empty:
    # Find numeric fields by checking data types (all columns are identified by @id)
    numeric_field_id = None
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric fields detected in this record set.")
    else:
        print(f"Using numeric field '@id': {numeric_field_id}")

        # Filter records where the value is above a threshold
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()

        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Find a categorical/grouping field (try string/object dtype or with low cardinality)
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == 'object':
                if df[col].nunique() < df.shape[0] // 2:
                    group_field_id = col
                    break
        if group_field_id:
            print(f"Grouping by field '@id': {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id, dropna=False)[numeric_field_id].mean().to_frame()
            display(grouped_df.head())
        else:
            print("No suitable grouping field detected.")
else:
    print("DataFrame is empty or not loaded.")

## 5. Visualization

Visualize data distributions or relationships between fields. The example below creates a histogram for the selected numeric field and a boxplot grouped by the group field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and numeric_field_id is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot grouped by group_field_id if such a field exists
    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to load, inspect, and analyze the FAIR^2 dataset defined by a Croissant schema. All dataset references, including record sets, fields, and columns, used their unique `@id` as required for reliable data handling. 

- The dataset structure (record sets, fields, columns) was explored and summarized.
- Data was loaded dynamically into DataFrames, allowing further Python analysis.
- Basic exploratory analysis and visualization enabled a deeper understanding of the dataset's structure and content.

**Next steps:** Apply custom domain-specific analysis, build predictive models, and refer to the dataset documentation and `@id` field mapping for accurate interpretations.